# JSON - Value Errors

Introduce seeded out-of-bounds or incorrectly scaled numeric measurements in one array.

**Instructions:** Use `prepare_data_json.ipynb` to create a clean `data/weather.json` first. This notebook overwrites it; restore the clean source before rerunning.

In [ ]:
from pathlib import Path
from random import Random
import copy
import json

import pandas as pd

# Configuration
DATA_PATH = Path("../../data/weather.json")
TIME_COLUMN = "time"
TIMEZONE = "America/Edmonton"
START_DATE = "2025-08-01"
END_DATE = "2025-09-01"  # Exclusive, local calendar dates.
SEED = 814205

# Value Errors
ERROR_TARGET_COLUMN = "relative_humidity_2m"
ERROR_KIND = "mixed"  # "out_of_bounds", "scaled", "mixed", "random".
ERROR_PLACEMENT = "mixed"
ERROR_RATE = (0.01, 0.03)  # Share of current eligible nonmissing/valid values changed.
ERROR_GROUPS = 3
ERROR_GROUPED_SHARE = 0.70
ERROR_VALID_MIN = 0
ERROR_VALID_MAX = 100
ERROR_SCALE_FACTOR = 1.5
ERROR_SCALE_OFFSET = 5


In [ ]:
# Shared JSON helpers: include once in a combined dirty_data.py.
def eligible_positions(document, *, time_column, timezone, start_date, end_date):
    hourly = document.get("hourly")
    if not isinstance(hourly, dict) or time_column not in hourly:
        raise ValueError("Expected a source-shaped document with hourly arrays.")
    if not all(isinstance(v, list) for v in hourly.values()):
        raise ValueError("All hourly fields must contain arrays.")
    if len({len(v) for v in hourly.values()}) != 1:
        raise ValueError("Hourly arrays must be aligned before ordinary mutations.")
    times = hourly[time_column]
    if not times or any(isinstance(t, bool) or not isinstance(t, (int, float)) for t in times):
        raise ValueError("Expected nonempty numeric Unix-second timestamps.")
    if any(a >= b for a, b in zip(times, times[1:])):
        raise ValueError("Expected unique increasing timestamps; apply duplicates last.")
    start = pd.Timestamp(start_date, tz=timezone).tz_convert("UTC").timestamp()
    end = pd.Timestamp(end_date, tz=timezone).tz_convert("UTC").timestamp()
    if start >= end:
        raise ValueError("Start date must precede end date.")
    positions = [i for i, t in enumerate(times) if start <= t < end]
    if not positions:
        raise ValueError("No observations in the selected interval.")
    return positions, start, end


def select_positions(n, rng, *, method, rate, n_groups, grouped_share):
    """Select a fraction of the current eligible records in ordered positions.

    Grouped blocks have at least two consecutive eligible records; mixed
    placement reserves at least one independently scattered selection.
    """
    if n < 1:
        raise ValueError("No eligible records.")
    if method not in ("scattered", "grouped", "mixed", "random"):
        raise ValueError("Unknown placement method.")
    if not 0 < rate[0] <= rate[1] < 1:
        raise ValueError("Rate must be between 0 and 1.")
    if n_groups < 1 or not 0 < grouped_share < 1:
        raise ValueError("Invalid group count or grouped share.")

    chosen = rng.choice(("scattered", "grouped", "mixed")) if method == "random" else method
    count = max(1, round(n * rng.uniform(*rate)))
    if chosen == "mixed" and count < 3:
        raise ValueError("Mixed placement needs two grouped and one scattered record.")
    grouped_count = (count if chosen == "grouped" else
                     max(2, min(count - 1, round(count * grouped_share))) if chosen == "mixed" else 0)
    selected = set()
    if grouped_count:
        if grouped_count < 2:
            raise ValueError("Grouped placement requires at least two affected records.")
        groups = min(n_groups, grouped_count // 2)
        if n - grouped_count < groups - 1:
            raise ValueError("Insufficient eligible records for separate groups.")
        length, extra = divmod(grouped_count, groups)
        lengths = [length + (i < extra) for i in range(groups)]
        rng.shuffle(lengths)
        slack = n - grouped_count - (groups - 1)
        dividers = sorted(rng.sample(range(slack + groups), groups))
        gaps = ([dividers[0]] +
                [dividers[i] - dividers[i - 1] - 1 for i in range(1, groups)] +
                [slack + groups - 1 - dividers[-1]])
        cursor = gaps[0]
        for i, block_length in enumerate(lengths):
            selected.update(range(cursor, cursor + block_length))
            if i < groups - 1:
                cursor += block_length + 1 + gaps[i + 1]

    scattered_count = count - grouped_count
    if scattered_count:
        remaining = [i for i in range(n) if i not in selected]
        selected.update(rng.sample(remaining, k=scattered_count))
    if len(selected) != count:
        raise AssertionError("Selected count differs from the intended count.")
    return sorted(selected), chosen


In [ ]:
def add_value_errors_json(
    document, rng, *, time_column, timezone, start_date, end_date,
    target_column, error_kind, placement, error_rate, n_groups,
    grouped_share, valid_min, valid_max, scale_factor, scale_offset,
):
    positions, _, _ = eligible_positions(
        document, time_column=time_column, timezone=timezone,
        start_date=start_date, end_date=end_date,
    )
    hourly = document["hourly"]
    if target_column not in hourly or target_column == time_column:
        raise ValueError("Target must be an existing measurement array.")
    values = hourly[target_column]
    # Other corruption functions may already have introduced nulls: leave them untouched.
    positions = [i for i in positions if values[i] is not None]
    if any(isinstance(values[i], bool) or not isinstance(values[i], (int, float)) for i in positions):
        raise ValueError("This numeric mutation requires numerical measurements.")
    if valid_min >= valid_max or scale_factor == 1 and scale_offset == 0:
        raise ValueError("Invalid bounds or identity scaling.")
    if error_kind not in ("out_of_bounds", "scaled", "mixed", "random"):
        raise ValueError("Unknown error kind.")
    kind = rng.choice(("out_of_bounds", "scaled", "mixed")) if error_kind == "random" else error_kind
    selected, _ = select_positions(
        len(positions), rng, method=placement, rate=error_rate,
        n_groups=n_groups, grouped_share=grouped_share,
    )
    if kind == "mixed" and len(selected) < 2:
        raise ValueError("Mixed errors need at least two affected values.")
    n_bounds = (len(selected) if kind == "out_of_bounds" else
                0 if kind == "scaled" else max(1, len(selected) // 2))
    bounds_positions = set(rng.sample(selected, k=n_bounds))
    dirty = copy.deepcopy(document)
    for i in selected:
        pos = positions[i]
        old = values[pos]
        if i in bounds_positions:
            new = valid_min - 1 if rng.random() < 0.5 else valid_max + 1
        else:
            new = old * scale_factor + scale_offset
        if new == old or not isinstance(new, (int, float)):
            raise ValueError("Mutation did not create a different numeric value.")
        dirty["hourly"][target_column][pos] = new
    return dirty


In [ ]:
# Small, reproducible source-shaped trial (no network required).
times = pd.date_range("2025-08-01", periods=24*14, freq="h", tz=TIMEZONE).tz_convert("UTC")
trial = {
    "latitude": 51.04, "timezone": "GMT", "hourly_units": {"temperature_2m": "°C"},
    "hourly": {
        "time": [int(t.timestamp()) for t in times],
        "temperature_2m": [float(i % 24) for i in range(len(times))],
        "relative_humidity_2m": [float(40 + i % 50) for i in range(len(times))],
    },
}

trial_dirty = add_value_errors_json(
    trial, Random(SEED), time_column=TIME_COLUMN, timezone=TIMEZONE,
    start_date=START_DATE, end_date="2025-08-15",
    target_column=ERROR_TARGET_COLUMN, error_kind=ERROR_KIND, placement=ERROR_PLACEMENT,
    error_rate=ERROR_RATE, n_groups=ERROR_GROUPS, grouped_share=ERROR_GROUPED_SHARE,
    valid_min=ERROR_VALID_MIN, valid_max=ERROR_VALID_MAX,
    scale_factor=ERROR_SCALE_FACTOR, scale_offset=ERROR_SCALE_OFFSET,
)
assert trial_dirty["latitude"] == trial["latitude"]
assert len({len(v) for v in trial_dirty["hourly"].values()}) == 1
assert trial_dirty["hourly"][ERROR_TARGET_COLUMN] != trial["hourly"][ERROR_TARGET_COLUMN]
trial_dirty["hourly"][TIME_COLUMN][:8]

In [ ]:
# Apply to the prepared JSON (this overwrites weather.json).
if DATA_PATH.exists():
    with DATA_PATH.open("r", encoding="utf-8") as f:
        clean = json.load(f)
    rng = Random(SEED)
    dirty = add_value_errors_json(
        clean, rng, time_column=TIME_COLUMN, timezone=TIMEZONE,
        start_date=START_DATE, end_date=END_DATE,
        target_column=ERROR_TARGET_COLUMN, error_kind=ERROR_KIND, placement=ERROR_PLACEMENT,
        error_rate=ERROR_RATE, n_groups=ERROR_GROUPS, grouped_share=ERROR_GROUPED_SHARE,
        valid_min=ERROR_VALID_MIN, valid_max=ERROR_VALID_MAX,
        scale_factor=ERROR_SCALE_FACTOR, scale_offset=ERROR_SCALE_OFFSET,
    )
    serialized = json.dumps(dirty, allow_nan=False)
    DATA_PATH.write_text(serialized, encoding="utf-8")
    saved = json.loads(DATA_PATH.read_text(encoding="utf-8"))
else:
    clean, dirty, saved = trial, trial_dirty, trial_dirty
    print("Prepared data/weather.json not found: showing controlled trial only.")


In [ ]:
# Inspect the document as an analytical table.
df = pd.DataFrame(saved["hourly"])
print("Shape:", df.shape)
print("Any nulls:", df.isna().any().any())
print("Nulls by field:", df.isna().sum().to_dict())
print("Duplicate times:", df[TIME_COLUMN].duplicated().sum())
df.head()

In [ ]:
# Developer-side verification against the known clean document.
assert {k: v for k, v in saved.items() if k != "hourly"} == {k: v for k, v in clean.items() if k != "hourly"}
assert len({len(v) for v in saved["hourly"].values()}) == 1
assert saved["hourly"][TIME_COLUMN] == clean["hourly"][TIME_COLUMN]
for field in clean["hourly"]:
    if field != ERROR_TARGET_COLUMN:
        assert saved["hourly"][field] == clean["hourly"][field]
changed = [i for i,(a,b) in enumerate(zip(clean["hourly"][ERROR_TARGET_COLUMN],saved["hourly"][ERROR_TARGET_COLUMN])) if a != b]
assert changed
print("Changed measurements:", len(changed))


In [ ]:
# Developer diagnostic: show where values actually changed.
local = pd.to_datetime(clean["hourly"][TIME_COLUMN], unit="s", utc=True).tz_convert(TIMEZONE)
changed = [a != b for a,b in zip(clean["hourly"][ERROR_TARGET_COLUMN], saved["hourly"][ERROR_TARGET_COLUMN])]
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(local, clean["hourly"][ERROR_TARGET_COLUMN], linewidth=0.8, label="Clean")
ax.scatter(local[changed], [saved["hourly"][ERROR_TARGET_COLUMN][i] for i,v in enumerate(changed) if v], s=12, label="Changed")
ax.set_title(f"Value errors: {ERROR_TARGET_COLUMN}")
ax.legend()
plt.show()
